# 📚 Time Series Fundamentals: Decomposition, Smoothing & Forecasting
> Section 19 of the Statistics & A/B Testing curriculum — Part 5: Business Application & Interview Prep continues  
> 통계 & A/B 테스트 커리큘럼 섹션 19 — Part 5: 비즈니스 응용 & 면접 대비, 계속

---
# 🎯 Learning Objective (학습 목표)
Today I want to learn / 오늘 배우고 싶은 것:
- [x] Decompose a time series into trend, seasonality, and residual, and quantify a peak vs. off-season swing<br>시계열을 추세·계절성·잔차로 분해하고, 성수기 대 비수기 편차를 정량화하기
- [x] Compute moving averages at different window sizes and measure the noise-reduction tradeoff<br>다양한 윈도우 크기로 이동평균을 계산하고 노이즈 감소 트레이드오프를 측정하기
- [x] Use autocorrelation (ACF) to discover a hidden periodic cycle, and forecast with Holt-Winters exponential smoothing plus a simulation-based prediction interval<br>자기상관(ACF)으로 숨겨진 주기적 패턴을 발견하고, Holt-Winters 지수평활과 시뮬레이션 기반 예측구간으로 예측하기

---
# 🧠 Concept (개념)

## What is it? (이것은 무엇인가?)
*(Explain it in your own words.)*

**EN:** A time series is data indexed by time (daily visits, monthly revenue) where the ORDER of observations carries information that a plain unordered sample (Sections 1-10) throws away. Classical decomposition splits a series into three components — trend (long-run direction), seasonality (a fixed, repeating pattern like weekly or yearly cycles), and residual (whatever noise is left over) — either additively (Y = trend + seasonal + residual) or multiplicatively (Y = trend × seasonal × residual), depending on whether the seasonal swing stays a constant SIZE or a constant PERCENTAGE as the trend grows.

**KR:** 시계열은 시간으로 색인된 데이터(일별 방문자 수, 월별 매출)이며, 관측치의 순서 자체가 정보를 담고 있다는 점에서 순서 없는 일반 표본(섹션 1~10)과 다릅니다. 고전적 분해는 시계열을 세 요소로 나눕니다 -- 추세(장기적인 방향), 계절성(주간·연간 주기처럼 고정되어 반복되는 패턴), 잔차(남은 노이즈) -- 계절적 편차가 추세가 커져도 일정한 '크기'로 유지되는지, 일정한 '비율'로 유지되는지에 따라 덧셈적(Y = 추세 + 계절성 + 잔차)이거나 곱셈적(Y = 추세 × 계절성 × 잔차)입니다.

### Three tools, three questions (세 가지 도구, 세 가지 질문)

| Tool / 도구 | Question it answers / 답하는 질문 |
|---|---|
| Decomposition<br>분해 | How much of this month's number is trend, how much is seasonal, how much is noise?<br>이번 달 숫자 중 추세는 얼마, 계절성은 얼마, 노이즈는 얼마인가? |
| Moving average<br>이동평균 | If I smooth out the noise, what does the underlying level actually look like?<br>노이즈를 걷어내면 실제 수준은 어떻게 보이는가? |
| ACF (autocorrelation)<br>ACF(자기상관) | Is there a repeating cycle in this data, and how long is it?<br>이 데이터에 반복되는 주기가 있는가, 그 길이는 얼마인가? |
| Holt-Winters smoothing<br>Holt-Winters 지수평활 | Given the trend and seasonality so far, what happens next — and how confident should I be?<br>지금까지의 추세와 계절성을 감안하면 다음에 무슨 일이 일어나는가 -- 얼마나 확신할 수 있는가? |

## Why do we use it? (왜 사용하는가?)
*(When is it useful?)*

**EN:** Without decomposition, a manager comparing December revenue to November sees a jump and might credit a recent campaign — when Example 1 will show the swing is simply the same seasonal pattern that repeats every year. Moving averages and ACF exist because raw daily/monthly numbers are noisy enough that eyeballing a line chart can't reliably answer "is this trending up?" or "is there a weekly cycle?" — these tools answer both with a number, not a guess.

**KR:** 분해 없이는, 12월 매출을 11월과 비교하는 관리자가 상승을 보고 최근 캠페인 덕분이라고 생각할 수 있습니다 -- 하지만 예제 1이 보여주듯 그 편차는 매년 반복되는 같은 계절적 패턴일 뿐입니다. 이동평균과 ACF가 존재하는 이유는 원본 일별·월별 숫자가 노이즈로 가득해서, 선 그래프를 눈으로만 보고는 "이게 상승 추세인가?"나 "주간 주기가 있는가?"에 신뢰성 있게 답할 수 없기 때문입니다 -- 이 도구들은 추측이 아니라 숫자로 답합니다.

## When is it used in Business Analytics? (비즈니스 분석에서 언제 사용되는가?)
*(Real-world use case)*

**EN:** Forecasting next quarter's revenue for a budget plan, explaining why a metric always dips in February before anyone panics, smoothing a noisy daily active-user count to spot a real trend change, or discovering that a KPI has an unexpected weekly rhythm even though no one labeled the data by day of week.

**KR:** 예산 계획을 위해 다음 분기 매출을 예측하거나, 아무도 놀라기 전에 어떤 지표가 왜 항상 2월에 하락하는지 설명하거나, 노이즈가 많은 일별 활성 사용자 수를 매끄럽게 해서 진짜 추세 변화를 발견하거나, 요일별로 데이터에 라벨을 붙이지 않았는데도 KPI에 예상치 못한 주간 리듬이 있다는 것을 발견하는 경우.

---
# 📝 Syntax (문법)

## Basic Syntax (기본 문법)

In [1]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose

dates = pd.date_range("2022-01-01", periods=36, freq="MS")
values = pd.Series(np.random.normal(100, 10, 36), index=dates)

result = seasonal_decompose(values, model="additive", period=12)
print(result.trend.dropna().head(3))
print(result.seasonal.head(3))
print(result.resid.dropna().head(3))

2022-07-01    101.716657
2022-08-01    101.725936
2022-09-01    101.901788
Freq: MS, Name: trend, dtype: float64
2022-01-01    0.247627
2022-02-01    7.728213
2022-03-01   -0.343538
Freq: MS, Name: seasonal, dtype: float64
2022-07-01   -14.944573
2022-08-01     1.302411
2022-09-01     8.288925
Freq: MS, Name: resid, dtype: float64


## Common Variations (자주 쓰는 변형)

In [2]:
# Moving average (pandas) and autocorrelation (statsmodels) are the other two core tools
# 이동평균(pandas)과 자기상관(statsmodels)이 나머지 두 핵심 도구
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.holtwinters import ExponentialSmoothing

moving_avg = values.rolling(window=3).mean()
autocorr = acf(values, nlags=12)

hw_model = ExponentialSmoothing(values, trend="add", seasonal="add", seasonal_periods=12).fit()
forecast = hw_model.forecast(3)
print(forecast)

2025-01-01    104.770137
2025-02-01    109.846131
2025-03-01    103.411549
Freq: MS, dtype: float64


---
# 🧪 Small Examples (예제)

## Example 1 — Decomposition: Quantifying Peak vs. Off-Season Sales (분해: 성수기 대 비수기 매출 정량화)

In [3]:
# Business question: leadership wants to know exactly how much of December's
# revenue jump is a real, repeatable seasonal effect vs. random noise --
# not just "December looked good."
# 비즈니스 질문: 경영진은 12월 매출 상승이 정확히 얼마나 진짜 반복되는
# 계절 효과이고 얼마나 무작위 노이즈인지 알고 싶습니다 -- "12월이 좋았다"가
# 아니라.

np.random.seed(42)
n_months = 36
months = pd.date_range("2022-01-01", periods=n_months, freq="MS")
trend = np.linspace(50000, 80000, n_months)
seasonal_pattern = np.array([-8000, -9000, -3000, 0, 2000, 3000, 2000, 1000, 2000, 5000, 12000, 15000])
seasonal = np.tile(seasonal_pattern, n_months // 12 + 1)[:n_months]
noise = np.random.normal(0, 2000, n_months)
sales = trend + seasonal + noise
ts_sales = pd.Series(sales, index=months)

result = seasonal_decompose(ts_sales, model="additive", period=12)
seasonal_by_month = result.seasonal[:12]
avg_level = result.trend.dropna().mean()
peak_month = seasonal_by_month.idxmax()
trough_month = seasonal_by_month.idxmin()

print(f"Average trend level: {avg_level:,.0f}")
print(f"Peak month: {peak_month.strftime('%B')} "
      f"(seasonal effect +{seasonal_by_month.max():,.0f}, {seasonal_by_month.max() / avg_level:+.1%} of average level)")
print(f"Trough month: {trough_month.strftime('%B')} "
      f"(seasonal effect {seasonal_by_month.min():,.0f}, {seasonal_by_month.min() / avg_level:+.1%} of average level)")
print()
print("This swing repeats EVERY year at the same calendar months, regardless of")
print("how much the underlying trend has grown -- it's not December's campaign,")
print("it's the seasonal component the same as every prior December.")
print("이 편차는 기저 추세가 얼마나 성장했는지와 무관하게 매년 같은 달에")
print("반복됩니다 -- 12월 캠페인 덕분이 아니라, 이전의 모든 12월과 같은")
print("계절 성분일 뿐입니다.")

Average trend level: 66,210
Peak month: December (seasonal effect +11,934, +18.0% of average level)
Trough month: February (seasonal effect -11,839, -17.9% of average level)

This swing repeats EVERY year at the same calendar months, regardless of
how much the underlying trend has grown -- it's not December's campaign,
it's the seasonal component the same as every prior December.
이 편차는 기저 추세가 얼마나 성장했는지와 무관하게 매년 같은 달에
반복됩니다 -- 12월 캠페인 덕분이 아니라, 이전의 모든 12월과 같은
계절 성분일 뿐입니다.


## Example 2 — Moving Averages: Window Size vs. Noise Reduction (이동평균: 윈도우 크기 대 노이즈 감소)

In [4]:
# Business question: daily website visits are noisy enough that a single day's
# drop looks alarming. How much does a moving average actually smooth things
# out, and does a bigger window always win?
# 비즈니스 질문: 일별 웹사이트 방문자 수는 노이즈가 많아서 하루의 하락이
# 걱정스러워 보입니다. 이동평균이 실제로 얼마나 매끄럽게 하며, 윈도우가
# 클수록 항상 더 좋을까요?

np.random.seed(7)
n_days = 180
true_trend = np.linspace(1000, 1300, n_days)
daily_noise = np.random.normal(0, 150, n_days)
daily_visits = true_trend + daily_noise
ts_visits = pd.Series(daily_visits, index=pd.date_range("2025-01-01", periods=n_days))
true_trend_series = pd.Series(true_trend, index=ts_visits.index)

raw_noise_std = (ts_visits - true_trend_series).std()
print(f"Raw day-to-day noise (std vs. true trend): {raw_noise_std:.1f} visits")
print()

for window in [3, 7, 14, 30]:
    moving_avg = ts_visits.rolling(window).mean().dropna()
    aligned_trend = true_trend_series.loc[moving_avg.index]
    residual_std = (moving_avg - aligned_trend).std()
    reduction_pct = (1 - residual_std / raw_noise_std)
    print(f"window={window:2d} days: residual std={residual_std:6.1f}  "
          f"(noise reduced {reduction_pct:.1%})")

print()
print("Bigger windows smooth more noise, but Example 4's forecast will show the")
print("tradeoff: a big moving average also reacts more SLOWLY to a genuine trend")
print("change -- there's no free lunch, only a size/lag tradeoff to choose.")
print("윈도우가 클수록 노이즈를 더 많이 없애지만, 예제 4의 예측이 보여주듯")
print("트레이드오프가 있습니다: 큰 이동평균은 진짜 추세 변화에도 더 느리게")
print("반응합니다 -- 공짜 점심은 없고, 선택해야 할 크기·지연 트레이드오프만")
print("있습니다.")

Raw day-to-day noise (std vs. true trend): 146.7 visits

window= 3 days: residual std=  83.3  (noise reduced 43.2%)
window= 7 days: residual std=  55.4  (noise reduced 62.2%)
window=14 days: residual std=  40.0  (noise reduced 72.7%)
window=30 days: residual std=  26.8  (noise reduced 81.7%)

Bigger windows smooth more noise, but Example 4's forecast will show the
tradeoff: a big moving average also reacts more SLOWLY to a genuine trend
change -- there's no free lunch, only a size/lag tradeoff to choose.
윈도우가 클수록 노이즈를 더 많이 없애지만, 예제 4의 예측이 보여주듯
트레이드오프가 있습니다: 큰 이동평균은 진짜 추세 변화에도 더 느리게
반응합니다 -- 공짜 점심은 없고, 선택해야 할 크기·지연 트레이드오프만
있습니다.


## Example 3 — Autocorrelation (ACF): Finding a 7-Day Cycle With No Day-of-Week Column (자기상관: 요일 정보 없이 7일 주기 발견)

In [5]:
# Business question: someone hands you a raw series of daily website visits
# with no date-of-week label. Is there a weekly pattern hiding in it, and how
# would you PROVE it rather than just eyeballing the chart?
# 비즈니스 질문: 누군가 요일 라벨이 없는 일별 웹사이트 방문자 원본 시계열을
# 건네줍니다. 그 안에 주간 패턴이 숨어 있을까요, 차트를 눈으로 보는 대신
# 어떻게 증명할까요?

np.random.seed(15)
n_days2 = 140
weekly_pattern = np.array([100, 95, 90, 92, 110, 180, 160])  # unlabeled -- just 7 repeating numbers
base = np.tile(weekly_pattern, n_days2 // 7 + 1)[:n_days2]
mild_trend = np.linspace(0, 20, n_days2)
noise2 = np.random.normal(0, 8, n_days2)
website_visits = pd.Series(base + mild_trend + noise2)

acf_values = acf(website_visits, nlags=21)
top_lags_desc = (np.argsort(acf_values[1:15])[::-1][:3] + 1).tolist()

print("Autocorrelation at each lag (day 1 to day 14):")
for lag in range(1, 15):
    marker = "  <-- local peak" if lag in (7, 14) else ""
    print(f"  lag {lag:2d} day(s): {acf_values[lag]:+.3f}{marker}")

strongest_lag = top_lags_desc[0]
print()
print(f"Top 3 lags by ACF strength (highest to lowest): {top_lags_desc}")
print(f"Autocorrelation spikes sharply every {strongest_lag} days ({acf_values[7]:+.3f} at lag 7,")
print(f"{acf_values[14]:+.3f} at lag 14) -- without ever being told 'this is Saturday,'")
print(f"the ACF alone reveals a {strongest_lag}-day (weekly) cycle purely from the numbers.")
print(f"{strongest_lag}일마다 자기상관이 급격히 치솟습니다(lag 7에서 {acf_values[7]:+.3f},")
print(f"lag 14에서 {acf_values[14]:+.3f}) -- '이건 토요일이다'라는 말을 들은 적 없어도,")
print(f"ACF만으로 숫자에서 {strongest_lag}일(주간) 주기를 순수하게 밝혀냅니다.")

Autocorrelation at each lag (day 1 to day 14):
  lag  1 day(s): +0.443
  lag  2 day(s): -0.238
  lag  3 day(s): -0.505
  lag  4 day(s): -0.512
  lag  5 day(s): -0.241
  lag  6 day(s): +0.406
  lag  7 day(s): +0.900  <-- local peak
  lag  8 day(s): +0.418
  lag  9 day(s): -0.228
  lag 10 day(s): -0.489
  lag 11 day(s): -0.497
  lag 12 day(s): -0.246
  lag 13 day(s): +0.375
  lag 14 day(s): +0.844  <-- local peak

Top 3 lags by ACF strength (highest to lowest): [7, 14, 1]
Autocorrelation spikes sharply every 7 days (+0.900 at lag 7,
+0.844 at lag 14) -- without ever being told 'this is Saturday,'
the ACF alone reveals a 7-day (weekly) cycle purely from the numbers.
7일마다 자기상관이 급격히 치솟습니다(lag 7에서 +0.900,
lag 14에서 +0.844) -- '이건 토요일이다'라는 말을 들은 적 없어도,
ACF만으로 숫자에서 7일(주간) 주기를 순수하게 밝혀냅니다.


## Example 4 — Holt-Winters Forecast With a Simulation-Based Prediction Interval (Holt-Winters 예측과 시뮬레이션 기반 예측구간)

In [6]:
# Business question: leadership wants next 6 months of revenue for a budget
# plan -- not just a single point guess, but an honest range of how wrong
# that guess could reasonably be.
# 비즈니스 질문: 경영진은 예산 계획을 위해 향후 6개월 매출을 원합니다 --
# 단일 추정치가 아니라, 그 추정이 얼마나 틀릴 수 있는지에 대한 정직한
# 범위까지.

from statsmodels.tsa.holtwinters import ExponentialSmoothing

hw_model = ExponentialSmoothing(ts_sales, trend="add", seasonal="add", seasonal_periods=12).fit()
point_forecast = hw_model.forecast(6)

np.random.seed(99)
n_simulations = 1000
simulated_paths = hw_model.simulate(nsimulations=6, repetitions=n_simulations, error="add")
lower_bound = simulated_paths.quantile(0.025, axis=1)
upper_bound = simulated_paths.quantile(0.975, axis=1)

print(f"Holt-Winters point forecast + 95% prediction interval from {n_simulations} simulated paths:")
for i in range(6):
    month_label = point_forecast.index[i].strftime("%Y-%m")
    print(f"  {month_label}: point={point_forecast.iloc[i]:>9,.0f}  "
          f"95% PI=[{lower_bound.iloc[i]:>9,.0f}, {upper_bound.iloc[i]:>9,.0f}]")

print()
print("The interval WIDENS the further out the forecast goes -- exactly like")
print("Section 8-3's prediction interval, uncertainty compounds with distance")
print("from the data, whether that distance is in X or in time.")
print("예측구간은 시점이 멀어질수록 넓어집니다 -- 섹션 8-3의 예측구간과")
print("똑같이, 데이터로부터 멀어질수록 불확실성이 누적됩니다. 그 거리가")
print("X축에서든 시간에서든 마찬가지입니다.")

Holt-Winters point forecast + 95% prediction interval from 1000 simulated paths:
  2025-01: point=   71,889  95% PI=[   67,354,    76,354]
  2025-02: point=   70,496  95% PI=[   65,464,    75,596]
  2025-03: point=   77,473  95% PI=[   71,599,    83,463]
  2025-04: point=   83,798  95% PI=[   76,354,    90,685]
  2025-05: point=   84,547  95% PI=[   75,944,    92,767]
  2025-06: point=   87,527  95% PI=[   77,055,    97,411]

The interval WIDENS the further out the forecast goes -- exactly like
Section 8-3's prediction interval, uncertainty compounds with distance
from the data, whether that distance is in X or in time.
예측구간은 시점이 멀어질수록 넓어집니다 -- 섹션 8-3의 예측구간과
똑같이, 데이터로부터 멀어질수록 불확실성이 누적됩니다. 그 거리가
X축에서든 시간에서든 마찬가지입니다.


## Example 5 — Practice: Apply All Four Concepts (연습: 네 가지 개념 모두 적용하기)

Fill in every `________` blank below. The cell runs with **no errors** even if you leave blanks — it just prints a reminder instead of that part's result.

아래 `________` 빈칸을 모두 채워보세요. 빈칸을 두어도 셀은 **에러 없이** 실행되며, 그 부분의 결과 대신 안내 메시지만 출력됩니다.

In [ ]:
# ============================================================
# Practice -- fill in every ________ blank, then re-run the cell
# 연습 -- 아래 ________ 빈칸을 모두 채운 뒤 셀을 다시 실행하세요
# ============================================================
BLANK = "________"

def ready(*vals):
    return all(v != BLANK for v in vals)

# ---------- Part A: read a seasonal effect off a decomposed series ----------
np.random.seed(42)
n_months_a = 36
months_a = pd.date_range("2022-01-01", periods=n_months_a, freq="MS")
trend_a = np.linspace(50000, 80000, n_months_a)
seasonal_pattern_a = np.array([-8000, -9000, -3000, 0, 2000, 3000, 2000, 1000, 2000, 5000, 12000, 15000])
seasonal_a = np.tile(seasonal_pattern_a, n_months_a // 12 + 1)[:n_months_a]
sales_a = trend_a + seasonal_a + np.random.normal(0, 2000, n_months_a)
ts_a = pd.Series(sales_a, index=months_a)
result_a = seasonal_decompose(ts_a, model="additive", period=12)
seasonal_by_month_a = result_a.seasonal[:12]
seasonal_by_month_a.index = seasonal_by_month_a.index.strftime("%B")

month_name_a = "July"   # TODO: pick a full month name to look up, e.g. "July"

if ready(month_name_a) and month_name_a in seasonal_by_month_a.index:
    print(f"Seasonal effect for {month_name_a}: {seasonal_by_month_a[month_name_a]:,.0f}")
elif ready(month_name_a):
    print(f"'{month_name_a}' isn't a recognized month name -- try e.g. 'July'.")
else:
    print("Part A \u270f\ufe0f  Fill in month_name_a above, then re-run.")

# ---------- Part B: noise reduction at a chosen moving-average window ----------
np.random.seed(7)
n_days_b = 180
true_trend_b = np.linspace(1000, 1300, n_days_b)
visits_b = pd.Series(true_trend_b + np.random.normal(0, 150, n_days_b),
                      index=pd.date_range("2025-01-01", periods=n_days_b))
true_trend_series_b = pd.Series(true_trend_b, index=visits_b.index)
raw_noise_std_b = (visits_b - true_trend_series_b).std()

window_b = 14   # TODO: pick a moving-average window size in days, e.g. 14

if ready(window_b):
    ma_b = visits_b.rolling(window_b).mean().dropna()
    residual_std_b = (ma_b - true_trend_series_b.loc[ma_b.index]).std()
    reduction_b = 1 - residual_std_b / raw_noise_std_b
    print(f"window={window_b}: noise reduced {reduction_b:.1%} (residual std={residual_std_b:.1f})")
else:
    print("Part B \u270f\ufe0f  Fill in window_b above, then re-run.")

# ---------- Part C: read the ACF value at a chosen lag ----------
np.random.seed(15)
n_days_c = 140
weekly_pattern_c = np.array([100, 95, 90, 92, 110, 180, 160])
base_c = np.tile(weekly_pattern_c, n_days_c // 7 + 1)[:n_days_c]
visits_c = pd.Series(base_c + np.linspace(0, 20, n_days_c) + np.random.normal(0, 8, n_days_c))
acf_values_c = acf(visits_c, nlags=21)

lag_c = 7   # TODO: pick a lag (in days) to inspect, e.g. 7

if ready(lag_c):
    print(f"ACF at lag {lag_c}: {acf_values_c[lag_c]:+.3f}")
else:
    print("Part C \u270f\ufe0f  Fill in lag_c above (an integer number of days), then re-run.")

# ---------- Part D: match the concept to its section number ----------
concept_d = "a 95% interval widens the further out a forecast reaches"

section_number_d = 11   # TODO: which earlier section first introduced this idea? (a number)

if ready(section_number_d):
    correct_section = 11
    print(f"Concept: '{concept_d}'")
    print(f"Your answer: Section {section_number_d}")
    if section_number_d == correct_section:
        print("Correct!")
    else:
        print(f"Not quite -- this is from Section {correct_section} (Simple Linear Regression's prediction interval).")
else:
    print("Part D \u270f\ufe0f  Fill in section_number_d above, then re-run.")

Part A ✏️  Fill in month_name_a above, then re-run.
Part B ✏️  Fill in window_b above, then re-run.
Part C ✏️  Fill in lag_c above (an integer number of days), then re-run.
Part D ✏️  Fill in section_number_d above, then re-run.


---
# ⚠️ Common Mistakes (자주 하는 실수)

### Mistake 1 — Treating a Moving Average as a Forecast (이동평균을 예측으로 착각)
A moving average only describes the PAST — it has no mechanism to extrapolate forward, and as Example 2 showed, a larger window lags further behind the true current level. Dragging the last moving-average value forward as "next month's forecast" systematically underestimates any real ongoing trend.  
이동평균은 과거만 설명합니다 -- 미래로 외삽할 메커니즘이 없으며, 예제 2가 보여줬듯 윈도우가 클수록 실제 현재 수준보다 더 뒤처집니다. 마지막 이동평균 값을 그대로 "다음 달 예측치"로 끌고 오면 실제로 진행 중인 추세를 체계적으로 과소평가합니다.

✅ **Fix / 해결:** Use a moving average to describe/smooth the past; use a model built for forecasting (Example 4's Holt-Winters) to project forward.

### Mistake 2 — Assuming Additive Decomposition When the Pattern Is Multiplicative (패턴이 곱셈적인데 덧셈 분해를 가정)
If the seasonal swing grows in dollar terms as the trend grows (e.g., the holiday bump is always about +20% rather than a fixed +$15,000), forcing an additive decomposition produces a seasonal component that clearly doesn't fit the early, smaller-trend years as well as the later ones.  
계절적 편차가 추세가 커짐에 따라 금액 기준으로도 커진다면(예: 연말 상승폭이 고정된 +$15,000이 아니라 항상 약 +20%라면), 덧셈 분해를 강제하면 계절 성분이 추세가 작았던 초기 연도에는 후기만큼 잘 맞지 않게 됩니다.

✅ **Fix / 해결:** Plot the series first — if the seasonal swing's SIZE grows with the trend, use `model="multiplicative"`; if it stays a roughly constant absolute size, use `model="additive"` (Example 1's choice).

### Mistake 3 — Picking a Window Size Without Considering the Known Cycle Length (알려진 주기 길이를 고려하지 않고 윈도우 크기 선택)
As Example 3's ACF revealed a genuine 7-day cycle, a 3-day or 5-day moving average on that same data would partially smooth INTO the cycle rather than across a full one, leaving a wavy residual pattern instead of a clean trend line.  
예제 3의 ACF가 진짜 7일 주기를 밝혀냈듯, 같은 데이터에 3일이나 5일 이동평균을 쓰면 주기 전체를 가로지르는 대신 주기 중간을 매끄럽게 하게 되어, 깔끔한 추세선 대신 물결치는 잔차 패턴이 남습니다.

✅ **Fix / 해결:** Check the ACF for a dominant lag first (Example 3), then set the moving-average window to match that full cycle length (e.g., 7 for a weekly cycle).

### Mistake 4 — Correlating Two Seasonal Series and Calling It Causation (계절성이 있는 두 시계열을 상관시켜놓고 인과관계라고 부름)
Two series that both have a December peak (e.g., ice cream sales and swimming pool visits, or unrelated revenue lines that both spike at year-end) will show a strong Section 5 correlation purely because they share the SAME seasonal calendar, not because one drives the other.  
둘 다 12월에 정점을 찍는 두 시계열(예: 아이스크림 매출과 수영장 방문, 혹은 둘 다 연말에 급등하는 무관한 매출 항목)은 한쪽이 다른 쪽을 유발해서가 아니라 단지 같은 계절 캘린더를 공유하기 때문에 섹션 5의 강한 상관관계를 보입니다.

✅ **Fix / 해결:** Decompose both series first (Example 1) and compare their RESIDUALS or de-seasonalized trends, not the raw seasonal series, before trusting a correlation between two time series.

### Mistake 5 — Reporting Only a Point Forecast, Never an Interval (구간 없이 점 예측만 보고)
As Example 4 demonstrated, the same Holt-Winters model that outputs a single forecast number can also simulate hundreds of plausible future paths — presenting only the point forecast hides exactly how much that number could reasonably be wrong by, which is often the more decision-relevant piece of information.  
예제 4가 보여줬듯, 단일 예측 숫자를 출력하는 같은 Holt-Winters 모델은 수백 개의 그럴듯한 미래 경로도 시뮬레이션할 수 있습니다 -- 점 예측만 제시하면 그 숫자가 실제로 얼마나 틀릴 수 있는지가 가려지며, 이는 흔히 의사결정에 더 중요한 정보입니다.

✅ **Fix / 해결:** Always pair a forecast with a prediction interval (via simulation, as in Example 4, or an analytical formula where available), especially before it's used for a budget or staffing decision.

### Mistake 6 — Ignoring That a Forecast's Uncertainty Grows With Distance (예측의 불확실성이 거리에 따라 커진다는 것을 무시)
Treating month 1 and month 6 of a forecast as equally trustworthy ignores that Example 4's simulated interval widens substantially further out — a 6-month-ahead forecast is a fundamentally less certain claim than a 1-month-ahead one, even from the identical model.  
예측의 1개월 차와 6개월 차를 똑같이 신뢰할 수 있다고 취급하는 것은 예제 4의 시뮬레이션 구간이 시점이 멀어질수록 크게 넓어진다는 사실을 무시하는 것입니다 -- 동일한 모델에서 나왔더라도 6개월 후 예측은 1개월 후 예측보다 근본적으로 덜 확실한 주장입니다.

✅ **Fix / 해결:** Always check how the prediction interval's width changes across the forecast horizon, and communicate near-term forecasts with more confidence than far-term ones.

---
# 💡 Tips (팁)
Useful tips or shortcuts / 유용한 팁과 단축법

- `seasonal_decompose` needs at least two full cycles of data to estimate a seasonal component reliably (e.g., 24+ months for a yearly cycle) — running it on a short series produces a noisy, untrustworthy seasonal estimate.<br>seasonal_decompose가 계절 성분을 신뢰성 있게 추정하려면 최소 두 번의 전체 주기 데이터가 필요합니다(예: 연간 주기라면 24개월 이상) -- 짧은 시계열에 실행하면 노이즈가 많고 신뢰할 수 없는 계절 추정치가 나옵니다.
- `df.rolling(window).mean()` drops the first `window - 1` rows as `NaN` since there aren't enough prior points yet — always `.dropna()` or account for this before comparing lengths against the original series.<br>df.rolling(window).mean()은 아직 이전 데이터가 충분하지 않은 처음 window-1개 행을 NaN으로 남깁니다 -- 원본 시계열과 길이를 비교하기 전에 항상 .dropna()를 하거나 이를 감안하세요.
- In an ACF plot, a slowly-decaying pattern at every lag usually signals a trend that should be removed first, while sharp, isolated spikes at regular intervals (Example 3's lag 7, 14, 21) are the clean signature of true seasonality.<br>ACF 플롯에서 모든 lag에서 천천히 감쇠하는 패턴은 보통 먼저 제거해야 할 추세를 나타내고, 일정한 간격으로 나타나는 날카롭고 고립된 스파이크(예제 3의 lag 7, 14, 21)는 진짜 계절성의 깔끔한 특징입니다.
- `ExponentialSmoothing`'s `seasonal_periods` argument must match the TRUE cycle length exactly (12 for monthly/yearly, 7 for daily/weekly) — get this wrong and the model fits a seasonal pattern that doesn't exist.<br>ExponentialSmoothing의 seasonal_periods 인자는 실제 주기 길이와 정확히 일치해야 합니다(월별/연간이면 12, 일별/주간이면 7) -- 잘못 설정하면 모델이 존재하지 않는 계절 패턴을 적합하게 됩니다.
- `.simulate(nsimulations, repetitions=N)` is the practical way to get a prediction interval for exponential smoothing since there's no simple closed-form formula the way Section 11 has for linear regression — more repetitions give a smoother, more stable interval estimate.<br>.simulate(nsimulations, repetitions=N)은 지수평활의 예측구간을 구하는 실용적인 방법입니다. 섹션 11의 선형회귀처럼 간단한 닫힌 형태 공식이 없기 때문입니다 -- 반복 횟수가 많을수록 더 매끄럽고 안정적인 구간 추정치를 얻습니다.

---
# 🔗 Related Concepts (관련 개념)

```
Business Metrics & Cohort Analysis (Section 18)
        ↓
Time Series Fundamentals (Section 19)  ← you are here / 현재 위치
   decomposition, moving average, ACF, Holt-Winters forecasting
        ↓
Interview Case Studies & Capstone Review (Section 20)
```

*How is today's topic connected to other concepts?*

**EN:** Decomposition's residual component is really the same idea as Section 12's regression residual — whatever is left after the model (here, trend + seasonal) explains what it can, and Example 1's residual should look like the same roughly-normal noise Section 3's distributions describe. Autocorrelation (Example 3) is literally Section 5's Pearson correlation applied to a series and a lagged copy of itself, and Holt-Winters' simulation-based interval (Example 4) solves the same problem Section 8-3's prediction interval solved for regression — quantifying honest uncertainty about a single future value, not just the average one. Section 20's capstone case studies will occasionally lean on exactly this kind of decomposition instinct ("rule out seasonality before crediting a real change") as one of its structured-diagnosis steps.

**KR:** 분해의 잔차 성분은 사실 섹션 12의 회귀 잔차와 같은 개념입니다 -- 모델(여기서는 추세+계절성)이 설명할 수 있는 것을 설명하고 남은 것이며, 예제 1의 잔차는 섹션 3의 분포가 설명하는 것과 같은 대략 정규분포인 노이즈처럼 보여야 합니다. 자기상관(예제 3)은 문자 그대로 시계열과 그 자신의 지연된 복사본에 적용된 섹션 5의 피어슨 상관관계이며, Holt-Winters의 시뮬레이션 기반 구간(예제 4)은 섹션 8-3의 예측구간이 회귀에서 풀었던 것과 같은 문제 -- 평균이 아닌 미래의 단일 값에 대한 정직한 불확실성을 정량화하는 것 -- 를 풀어냅니다. 섹션 20의 캡스톤 케이스 스터디는 구조화된 진단 단계의 하나로 바로 이런 분해적 직관("진짜 변화라고 인정하기 전에 계절성부터 배제하라")에 종종 의존합니다.

---
# 💼 Business Example (비즈니스 예시)
*How would a Business Analyst use this?*

**Scenario / 시나리오:**

**EN:** Finance is building next year's budget and needs monthly revenue guidance for the next 6 months, plus a plain-language explanation of why revenue always dips in Q1 so nobody mistakes it for a real problem.

**KR:** 재무팀이 내년도 예산을 세우고 있으며, 향후 6개월의 월별 매출 가이던스와 왜 매출이 항상 1분기에 하락하는지에 대한 쉬운 설명이 필요합니다 -- 아무도 이를 진짜 문제로 오해하지 않도록.

**To-do / 할 일:**
- [x] Decompose historical revenue to quantify the Q1 seasonal dip<br>과거 매출을 분해해 1분기 계절적 하락을 정량화하기
- [x] Fit a Holt-Winters model and forecast the next 6 months<br>Holt-Winters 모델을 적합하고 향후 6개월 예측하기
- [x] Produce a 95% prediction interval via simulation for the budget range<br>예산 범위를 위해 시뮬레이션으로 95% 예측구간 산출하기
- [x] Write a one-paragraph explanation of the Q1 dip for non-technical stakeholders<br>비기술직 이해관계자를 위한 1분기 하락 설명 한 문단 작성하기

In [8]:
result_biz = seasonal_decompose(ts_sales, model="additive", period=12)
seasonal_by_month_biz = result_biz.seasonal[:12]
avg_level_biz = result_biz.trend.dropna().mean()
q1_months = ["January", "February", "March"]
seasonal_by_month_biz.index = seasonal_by_month_biz.index.strftime("%B")
q1_avg_effect = seasonal_by_month_biz[q1_months].mean()

hw_model_biz = ExponentialSmoothing(ts_sales, trend="add", seasonal="add", seasonal_periods=12).fit()
forecast_biz = hw_model_biz.forecast(6)

np.random.seed(2024)
sims_biz = hw_model_biz.simulate(nsimulations=6, repetitions=1000, error="add")
lower_biz = sims_biz.quantile(0.025, axis=1)
upper_biz = sims_biz.quantile(0.975, axis=1)

print(f"Q1 seasonal effect: {q1_avg_effect:,.0f} on average ({q1_avg_effect / avg_level_biz:+.1%} of the average level)")
print()
print("6-month revenue forecast with 95% prediction interval:")
for i in range(6):
    month_label = forecast_biz.index[i].strftime("%Y-%m")
    print(f"  {month_label}: point={forecast_biz.iloc[i]:>9,.0f}  "
          f"95% PI=[{lower_biz.iloc[i]:>9,.0f}, {upper_biz.iloc[i]:>9,.0f}]")

print()
print("Stakeholder-friendly explanation: Q1 revenue is expected to run about")
print(f"{abs(q1_avg_effect):,.0f} below the underlying trend every year -- this is a")
print("repeatable seasonal pattern seen in every prior January-March, not a sign")
print("anything is going wrong. Budget the Q1 dip in, and treat only a deviation")
print("from THIS pattern as something worth investigating.")
print(f"이해관계자용 설명: 1분기 매출은 매년 기저 추세보다 약 {abs(q1_avg_effect):,.0f}만큼")
print("낮게 나올 것으로 예상됩니다 -- 이는 이전의 모든 1~3월에서 나타난 반복되는")
print("계절 패턴이지, 무언가 잘못되고 있다는 신호가 아닙니다. 1분기 하락을")
print("예산에 반영하고, 오직 이 패턴에서 벗어나는 경우만 조사할 가치가 있는")
print("것으로 취급하세요.")

Q1 seasonal effect: -9,400 on average (-14.2% of the average level)

6-month revenue forecast with 95% prediction interval:
  2025-01: point=   71,889  95% PI=[   67,669,    76,464]
  2025-02: point=   70,496  95% PI=[   65,775,    75,479]
  2025-03: point=   77,473  95% PI=[   71,615,    83,022]
  2025-04: point=   83,798  95% PI=[   77,371,    90,745]
  2025-05: point=   84,547  95% PI=[   76,028,    93,563]
  2025-06: point=   87,527  95% PI=[   77,698,    97,405]

Stakeholder-friendly explanation: Q1 revenue is expected to run about
9,400 below the underlying trend every year -- this is a
repeatable seasonal pattern seen in every prior January-March, not a sign
anything is going wrong. Budget the Q1 dip in, and treat only a deviation
from THIS pattern as something worth investigating.
이해관계자용 설명: 1분기 매출은 매년 기저 추세보다 약 9,400만큼
낮게 나올 것으로 예상됩니다 -- 이는 이전의 모든 1~3월에서 나타난 반복되는
계절 패턴이지, 무언가 잘못되고 있다는 신호가 아닙니다. 1분기 하락을
예산에 반영하고, 오직 이 패턴에서 벗어나는 경우만 조사할 가치가 있는
것으로 취급하세요.


---
# 📝 Summary (요약)
*Write today's concept in 3~5 sentences.*

**EN:** Classical decomposition splits a time series into trend, seasonality, and residual, which lets Example 1 prove a December revenue jump is the same repeatable seasonal effect every year rather than crediting a one-off cause. Moving averages smooth out noise to reveal the underlying level, but Example 2 showed this comes with a size/lag tradeoff — a bigger window reduces noise more but reacts more slowly to a genuine change, and it can never extrapolate into a real forecast. Autocorrelation (ACF) is Pearson correlation applied to a series and its own lagged copy, and Example 3 showed it can discover a periodic cycle (a 7-day pattern) purely from the numbers, with no day-of-week label required. Holt-Winters exponential smoothing produces both a point forecast and, via simulation, a prediction interval that widens the further out it projects — exactly like Section 11's regression prediction interval, honest uncertainty grows with distance from the data, whether that distance is measured in X or in time.

**KR:** 고전적 분해는 시계열을 추세·계절성·잔차로 나누며, 이는 예제 1이 12월 매출 상승이 일회성 원인 덕분이 아니라 매년 반복되는 같은 계절 효과임을 증명하게 해줍니다. 이동평균은 노이즈를 매끄럽게 해 기저 수준을 드러내지만, 예제 2가 보여줬듯 여기에는 크기·지연 트레이드오프가 따릅니다 -- 윈도우가 클수록 노이즈는 더 줄지만 진짜 변화에 더 느리게 반응하며, 실제 예측으로 외삽할 수는 없습니다. 자기상관(ACF)은 시계열과 그 자신의 지연된 복사본에 적용된 피어슨 상관관계이며, 예제 3이 보여줬듯 요일 라벨 없이도 숫자만으로 주기적 패턴(7일 패턴)을 발견할 수 있습니다. Holt-Winters 지수평활은 점 예측과 함께, 시뮬레이션을 통해 시점이 멀어질수록 넓어지는 예측구간을 만들어냅니다 -- 섹션 11의 회귀 예측구간과 정확히 같이, 데이터로부터의 거리가 X축이든 시간이든 정직한 불확실성은 거리와 함께 커집니다.

---
# 📌 One Sentence Summary (한 문장 요약)
Today's topic in ONE sentence. / 오늘의 주제를 한 문장으로.

> **EN:** A time series decomposes into trend, seasonality, and residual — moving averages smooth it with a size/lag tradeoff, ACF finds a hidden cycle by correlating the series with its own lagged copy, and Holt-Winters forecasts forward with a simulation-based prediction interval that, like any forecast, widens the further out it reaches.

> **KR:** 시계열은 추세·계절성·잔차로 분해되며 -- 이동평균은 크기·지연 트레이드오프를 갖고 이를 매끄럽게 하고, ACF는 시계열을 자신의 지연된 복사본과 상관시켜 숨겨진 주기를 찾아내며, Holt-Winters는 어떤 예측이든 그렇듯 멀리 갈수록 넓어지는 시뮬레이션 기반 예측구간과 함께 미래를 예측합니다.

---
# ❓ Review Questions (복습 질문)

**Q1 (EN).** A manager sees December revenue is always higher than November and credits a recent campaign. What would you check first, and with which tool?
**Q1 (KR).** 관리자가 12월 매출이 항상 11월보다 높은 것을 보고 최근 캠페인 덕분이라고 생각합니다. 무엇을, 어떤 도구로 먼저 확인하겠습니까?

Check seasonality first by decomposing the series (seasonal decomposition) to see whether December is high every year, campaign or not.  
먼저 계절성을 확인합니다. 시계열 분해(seasonal decomposition)로 캠페인과 상관없이 매년 12월이 높은지 봅니다.

**Q2 (EN).** Why can't a moving average be dragged forward and used directly as next month's forecast?
**Q2 (KR).** 이동평균을 그대로 끌고 와서 다음 달 예측치로 바로 사용할 수 없는 이유는 무엇인가요?

A moving average lags behind the data and ignores trend and seasonality, so it only describes the past, not the next value.  
이동평균은 데이터보다 뒤처지고 추세와 계절성을 반영하지 않아서, 과거를 설명할 뿐 다음 값을 예측하지 못합니다.

**Q3 (EN).** You're handed a daily metric with no date-of-week column and asked whether it has a weekly pattern. Which tool answers this, and what would the output look like if the answer is yes?
**Q3 (KR).** 요일 열이 없는 일별 지표를 받고 주간 패턴이 있는지 질문받았습니다. 어떤 도구가 이에 답하며, 답이 '그렇다'라면 결과는 어떻게 보일까요?

Use autocorrelation (ACF); a weekly pattern shows as spikes at lags 7, 14, 21.  
자기상관(ACF)을 씁니다. 주간 패턴이 있으면 래그 7, 14, 21에서 뚜렷한 봉우리가 나타납니다.

**Q4 (EN).** Why does a Holt-Winters forecast's prediction interval get wider the further into the future it projects, and why does that matter for a budget decision?
**Q4 (KR).** Holt-Winters 예측의 예측구간이 미래로 갈수록 넓어지는 이유는 무엇이며, 이것이 예산 의사결정에서 왜 중요한가요?

Forecast errors accumulate with each step, so uncertainty grows; for budgets, plan with the range, not just the single forecast line.  
예측 오차가 단계마다 쌓여 불확실성이 커지기 때문입니다. 예산은 예측선 하나가 아니라 구간으로 계획해야 합니다.

**Q5 (EN).** Two unrelated business metrics both spike every December. Why might they show a strong Section 5 correlation, and what should you check before believing one causes the other?
**Q5 (KR).** 서로 무관한 두 비즈니스 지표가 둘 다 매년 12월에 급등합니다. 왜 섹션 5의 강한 상관관계를 보일 수 있으며, 한쪽이 다른 쪽을 유발한다고 믿기 전에 무엇을 확인해야 하나요?

Shared seasonality creates a spurious correlation, so check the correlation after removing trend and seasonality (for example on the residuals or differenced data) and look for a real mechanism.  
같은 계절성을 공유해서 가짜 상관이 생깁니다. 추세와 계절성을 제거한 뒤(잔차나 차분 데이터) 상관을 다시 보고, 실제 인과 경로가 있는지 확인해야 합니다.

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*
*🎉 Next up: Section 20 — Interview Case Studies & Capstone Review, the final section of the curriculum. / 다음: 섹션 20 — 면접 케이스 스터디 & 캡스톤 복습, 커리큘럼의 마지막 섹션입니다.*